## Multi Agent Design

#### Import libraries

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

load_dotenv(override=True)

#### Define Model

In [ ]:
#model = "gpt-4.1-nano"

# Alternatively, you can use a local model
client = AsyncOpenAI(base_url="http://localhost:11434/v1")
model = OpenAIChatCompletionsModel(model = "gpt-oss",openai_client= client)


#### Define Tools

In [ ]:
# --- Specialist tools ---
@function_tool
def calc_gc_content(dna_sequence: str) -> str:
    """Calculate GC content of a DNA sequence."""
    gc = dna_sequence.count('G') + dna_sequence.count('C')
    percent = 100 * gc / len(dna_sequence)
    return f"GC content is {percent:.2f}%."

@function_tool
def translate_protein(dna_sequence: str) -> str:
    """Translate DNA sequence to protein (mocked)."""
    return "Possible protein translations: MRR, MKT, ... (mocked result)"

@function_tool
def find_snps(dna_sequence: str) -> str:
    """Find known SNPs in the DNA sequence (mocked)."""
    return "Known SNPs: None found in this sequence (mocked result)"


#### Define Special Agents

In [ ]:
gc_agent = Agent(
    name="GCContentAgent",
    instructions="Given a DNA sequence, use your tool to calculate GC content.",
    model=model,
    tools=[calc_gc_content],
).as_tool("gc_content_agent", tool_description="Agent to calculate GC content.")

protein_agent = Agent(
    name="ProteinAgent",
    instructions="Given a DNA sequence, use your tool to translate it to all possible proteins.",
    model=model,
    tools=[translate_protein],
).as_tool("protein_agent", tool_description="Agent to translate DNA to protein.")

snp_agent = Agent(
    name="SNPAagent",
    instructions="Given a DNA sequence, use your tool to find known SNPs.",
    model=model,
    tools=[find_snps],
).as_tool("snp_agent", tool_description="Agent to find SNPs in DNA.")


#### Define Main agent

In [ ]:
# --- Main agent with access to specialist agents as tools ---
main_agent = Agent(
    name="CoordinatorAgent",
    instructions=(
        "You are a bioinformatics coordinator. Given a user request and a DNA sequence, "
        "decide which specialist agent to call: gc_content_agent, protein_agent, or snp_agent. "
        "Call the appropriate agent and return the result. Also mention the reasoning behind calling that agent. Output in 50 words or less"
    ),
    model=model,
    tools=[gc_agent, protein_agent, snp_agent],
)

#### Call the main agent for GC content

In [ ]:
from agents import trace
with trace("main_agent_pipeline"):
    # --- Example usage ---
    user_request = "What is the GC content of this sequence ATGCGGAATTCGCGTAAATGAATTCGCGT"
    result = await Runner.run(main_agent, user_request)
    print(result.final_output)